## Extract and Plot River discharge + Inundation from ECMWF ecLand-CaMaFlood

In [ ]:
# command to use in case grib file does not load
# need to be set before import metview
import os
os.environ["MARS_READANY_BUFFER_SIZE"] = "2147483648"  # "7464960400"  # "3732480200"  # "78930123"

# Redirect TMPDIR to SCRATCHDIR
os.environ["TMPDIR"] = os.environ.get("SCRATCHDIR")
!mkdir -p $TMPDIR

In [ ]:
import os
import metview as mv
import numpy as np 
import datetime

### Setup

In [ ]:
!rm -rf /perm/pad/flood_cases/*_flood.grb
!rm -rf /perm/pad/flood_cases/*_flood_globe.grb
!rm -rf o_flx_cmf.grb

In [ ]:
# specify experiment ID and parameters, see: https://confluence.ecmwf.int/display/~pajd/Offline+ECLAND+simulations
exp = "hvrl" #4km
exp = "ht0t" #9km
exp = "htnq" #25km
exp = "htnr" #25km
exp = "i05l" #4km
exp = "immh" #4km
exp = "ixzz" #4km
exp = "iwar" #4km
exp = "ixot" #4km
exp = "iyqu" #4km
exp = "j1ee" #1km
exp = "izay" #4km
exp = "iz46" #1km
exp = 'j1ee' #1km

parameter = ["235270","235275"]
flood_cases=["Germany","Lybia","Australia","Pakistan","SE_Asia","Thessaly"]
flood_cases=""

delta=24
stream="lwda"
stream="oper"

for flood_case in flood_cases:
# specify date and time
# specify extraction area
    if ( flood_case == "Germany"):
        yy=2021 ; mm=7
        date = datetime.date(year=yy, month=mm, day=1).isoformat() # Germany 15/7/2025
        step = [336] # Germany
        area = [44, 0., 54.,10.] # Germany

    if ( flood_case == "Lybia"):
        yy=2023 ; mm=9
        date = datetime.date(year=yy, month=mm, day=1).isoformat() # Lybia 12/9/2023
        step = [264] # Lybia
        area = [26, 20, 34, 25] # Lybia

    if ( flood_case == "Australia"):
        yy=2025 ; mm=3
        dates = datetime.date(year=yy, month=mm, day=1).isoformat() # Australia 29/3/2025
        steps = [672] # Australia
        steps = [480,"to",740,"by",delta]
        area = [-45, 110, -10, 155] # Australia

    if ( flood_case == "US"):
        yy=2022 ; mm=7
        date = datetime.date(year=yy, month=mm, day=1).isoformat() # US Flood Hurrican Catrina 23/8/2005
        step = [576] # Us
        area = [20, -90, 50, -70] # US

    if ( flood_case == "Pakistan"):
        yy=2019 ; mm=2
        yy=2022 ; mm=8
        date = datetime.date(year=yy, month=mm, day=1).isoformat() # Pakistan 21/2/2019
        step = [480] # Pakistan 21/2/2019
        step = [744] # Pakistan 28/8/2022
        area = [22, 66, 31, 72] # Pakistan

    if ( flood_case == "SE_Asia"):
        yy=2020 ; mm=7
        date = datetime.date(year=yy, month=mm, day=1).isoformat() # SE_Asia 2/7/2020
        step = [24] # SE_Asia
        area = [5, 60, 40, 95] # SE_Asia 

    if ( flood_case == "Thessaly"):
        yy=2023 ; mm=9
        date = datetime.date(year=yy, month=mm, day=1).isoformat() # Thessaly 10/9/2023
        step = [240] # Thessaly
        area = [35, 18, 42, 25] # Thessaly
        
#    dates = date
time = datetime.time(hour=0).isoformat()
time = 0
#    steps = [delta,"to",6,"by",delta]


if ( flood_cases=="" ):
    flood_case="US"
    area = [10, -140, 50, -70] # US

    flood_case="Europe"
    area = [30, -25, 85, 45] # Europe
    
    flood_case="Globe"
    area = [-90, -180, 90, 180] # Globe 

    flood_case="Indonesia"
    area = [-10, 90, 10, 115] # Indonesia 
    
    flood_case="SriLanka"
    area = [5, 70, 15, 85] # SriLanka 
    
    flood_case="Oregon"
    area = [44, -126, 50, -120]
    dates = [20251208,"to",20251211]

    flood_case="TonleSap"
    area= [15, 102, 10, 108]
    dates = [20265301,"to",20260302,"by",1]   
    dates = [20181015,"to",20181016]   
    
    flood_case="SouthAfrica"
    area = [3, 5, -35, 43]
    dates = [20260318,"to",20260318]   

    flood_case="Persia"
    area = [50, 35, 20, 65]
    area = [35, 43, 29, 50]
    dates = [20260331,'to',20260331]  

    flood_case="SouthItaly"
    area = [43, 12, 36, 19] #SouthItaly
    dates = 20260402
    
    flood_case = "Pakistan"
    area = [22, 66, 31, 72] # Pakistan
    
    flood_case = "Amazon"
    area = [20, 80, 30, 95]  # Amazon 
    dates = 20210616

    flood_case="Australia"
    area = [-45, 110, -10, 155] # Australia
    dates = 20250329
    
    flood_case = "Yangtze"
    area = [28, 105, 38, 125]  # Yangtze 
    dates = [20200722,'to',20200722]  
    
    steps = [delta,'to',24,'by',delta]
    steps = 24
    print("Extract",parameter,"from experiment",exp,"for case study",flood_case)


In [ ]:
mars_request = {
    'class': 'rd',
    'expver': exp,
    'stream': stream,
    'type': 'fc',
    'levtype': 'sfc',
    'date': dates,
    'time': time,
    'step': steps,
    'param': parameter,
    }
    
print (mars_request)
output="/perm/pad/flood_cases/"+flood_case+"_flood_globe.grb"
    
if os.path.exists(output):
    print("File "+output+" already exists!")
else:
    print("File "+output+" does not exist. Extracting and Saving")
    data = mv.retrieve(mars_request)
    data.describe()
    data.write(output)

In [ ]:
!echo wgrib2 /perm/pad/flood_cases/{flood_case}_flood_globe.grb -small_grib {area[1]}:{area[3]} {area[0]}:{area[2]} /perm/pad/flood_cases/{flood_case}_flood.grb
output="/perm/pad/flood_cases/"+flood_case+"_flood.grb"
!wgrib2 /perm/pad/flood_cases/{flood_case}_flood_globe.grb -small_grib {area[1]}:{area[3]} {area[0]}:{area[2]} /perm/pad/flood_cases/{flood_case}_flood.grb

In [ ]:
#!rm /perm/pad/flood_cases/bar
!ls /perm/pad/flood_cases/

#### Read in data

In [ ]:
mv.setoutput("jupyter", output_width=1600, output_font_scale=3, plot_widget=True)
#mv.setoutput(mv.png_output(output_name="river_3arcmin", output_width=1600, output_font_scale=3))

In [ ]:
# define coastlines
coastlines = mv.mcoast(
    map_coastline_resolution="high",
    map_coastline_sea_shade="on",
    map_coastline_sea_shade_colour="cyan",
    map_coastline_land_shade="on",
    map_coastline_land_shade_colour="cream",
    map_grid_frame_colour="grey",
    map_grid_frame_line_style="dash",
    map_grid_frame="off",
    map_coastline="on",
    map_label="off",
#    map_grid="off",
    )

view_earth = mv.geoview(
    map_projection="tpers",
    map_projection_view_latitude=40.0,
    map_projection_view_longitude=60.0,
    coastlines=coastlines,
    page_frame="off",
    subpage_frame="off",
    subpage_background_colour="black",
)

view_geo = mv.geoview(
        map_projection="geos",
        map_vertical_longitude=60.0,
        coastlines=coastlines,
        page_frame="off",
        subpage_frame="off",
)

view = mv.geoview(
    map_projection="robinson",
    subpage_y_position=14,
    subpage_y_length=86,
    coastlines=coastlines,
    page_frame="off",
    subpage_frame="off",
)

# define views
view_coast = mv.geoview(coastlines=coastlines)
view = mv.make_geoview(area="base")
view_nh = mv.make_geoview(area="north_pole")

view_ce = mv.make_geoview(area="central_europe")
view_na = mv.make_geoview(area="north_america")
view_ss = mv.make_geoview(area="south_america")
view_sa = mv.make_geoview(area="southern_asia")
view_eu = mv.make_geoview(area="europe")
view_ea = mv.make_geoview(area="eurasia")
view_aa = mv.make_geoview(area="australasia")
view_az = mv.make_geoview(area="australia")

view_eu_coast = mv.geoview(area_mode="name", area_name="europe", coastlines=coastlines)
view_alps = mv.geoview(map_area_definition="corners", area=[40, 0, 55, 20])
view_himalaya=mv.geoview(map_area_definition="corners", area=[10, 65, 45, 115])
view_australia=mv.geoview(map_area_definition="corners",area = [-45, 107, -10, 158])
view_pk=mv.geoview(map_area_definition="corners",area = [0, 50, 35, 90])
view_lyb=mv.geoview(map_area_definition="corners",area = [26, 20, 34, 25])
view_alps_details = mv.geoview(map_area_definition="corners", area=[43, 4, 49, 17])

view_area=mv.geoview(map_area_definition="corners",area = area ,coastlines=coastlines)

In [ ]:
fc = mv.read(output)
data= mv.read (data=fc, step=steps)

In [ ]:
data.describe()

In [ ]:
#Plot

In [ ]:
conr = mv.mcont(
legend="on",
contour_label="off",
contour_label_height=0.2,
contour="off",
contour_method="linear",
contour_level_selection_type="level_list",
contour_reference_level=0,
contour_shade="on",
contour_shade_min_level=0.01,
contour_shade_max_level=250000,
CONTOUR_LEVEL_LIST = [ 5,10,50,100,500,1000,5000,10000,50000],
contour_shade_method="area_fill",
contour_shade_technique = "grid_shading",
contour_line_colour="black",
contour_highlight="off",
contour_label_colour="gold",
contour_hilo="off",
grib_scaling_of_retrieved_fields="off",
CONTOUR_SHADE_MAX_LEVEL_COLOUR = "RED",
CONTOUR_SHADE_MIN_LEVEL_COLOUR = "WHITE",
)

conm = mv.mcont(
legend="on",
contour_label="off",
contour_label_height=0.2,
contour="off",
contour_method="linear",
contour_level_selection_type="level_list",
contour_reference_level=0,
contour_shade="on",
contour_shade_min_level=-50000,
contour_shade_max_level=-0.01,
CONTOUR_LEVEL_LIST = [ -5000,-1000,-500,-100,-50,-10,-5],
contour_shade_method="area_fill",
contour_shade_technique = "grid_shading",
contour_line_colour="black",
contour_highlight="off",
contour_label_colour="gold",
contour_hilo="off",
grib_scaling_of_retrieved_fields="off",
CONTOUR_SHADE_MAX_LEVEL_COLOUR = "WHITE",
CONTOUR_SHADE_MIN_LEVEL_COLOUR = "BLUE",
)

Inundation = mv.mcont(
LEGEND = "ON",
contour_label_height=0.2,
CONTOUR = "OFF",
CONTOUR_LEVEL_SELECTION_TYPE = "LEVEL_LIST",
CONTOUR_MAX_LEVEL = 1.2,
CONTOUR_MIN_LEVEL = 0.01,
CONTOUR_SHADE_MAX_LEVEL = 1.2,
CONTOUR_SHADE_MIN_LEVEL = 0.01,
CONTOUR_LEVEL_LIST = [ 0.01,0.02,0.05,0.1,0.2,0.5,0.7,0.9,1.1],
CONTOUR_LABEL = "OFF",
CONTOUR_SHADE = "ON",
CONTOUR_SHADE_MAX_LEVEL_COLOUR = "NAVY",
CONTOUR_SHADE_MIN_LEVEL_COLOUR = "GREY",
CONTOUR_SHADE_TECHNIQUE = "grid_shading"
)

In [ ]:
t = mv.grib_get(fc, ["dataDate"])[0][0]
step = mv.grib_get(fc, ["step"])[0][0]
st=int(step)/24
datet=str(int(t)+int(st))

In [ ]:
font_size=0.2
# set up the position and properties of the legend
legend = mv.mlegend(
    legend_title="ON",
    legend_box_mode="POSITIONAL",
    legend_box_x_position=2.20,
    legend_box_x_length=2.00,
    legend_box_y_position=1.00,
    legend_box_y_length=8.00,
    legend_display_type="CONTINUOUS",
    legend_border="OFF",
    legend_text_font_size=font_size,
    legend_text_colour="black",
#    legend_text_colour="white",
    legend_title_text="River discharge (m3/s)   Inundated area (-)",
    legend_title_orientation="HORIZONTAL",
)

In [ ]:
pos_title = mv.mtext(text_line_1         = "IFS River discharge and inundation", text_mode           = "positional",
    text_box_x_position = 0, text_box_y_position = 17.4, text_box_x_length   = 8.2, text_box_y_length   = 1,
    text_font_size=font_size, text_colour="black")

pos_title2 = mv.mtext(text_line_1  = "Date "+datet, text_mode = "positional",
    text_box_x_position = 0, text_box_y_position = 16.8, text_box_x_length   = 7.5, text_box_y_length   = 1,
    text_font_size=font_size, text_colour="black")

notitle = mv.mtext(text_lines=[" "], text_font_size=font_size)

In [ ]:
mv.setoutput("jupyter", output_width=1600, output_font_scale=3, plot_widget=True)
mv.plot(data[0],conm,data[0],conr,data[1],Inundation, view_area, legend,notitle,pos_title,pos_title2)

In [ ]:
mv.setoutput(mv.png_output(output_name="CMF_1arcmin_"+flood_case+".png", output_width=1600, output_font_scale=4))
mv.plot(data[0],conm,data[0],conr,data[1],Inundation, view_area, legend,notitle,pos_title,pos_title2)

In [ ]:
# Animation

In [ ]:
font_size=0.9
# set up the position and properties of the legend
legend = mv.mlegend(
    legend_title="ON",
    legend_box_mode="POSITIONAL",
    legend_box_x_position=2.20,
    legend_box_x_length=2.00,
    legend_box_y_position=1.00,
    legend_box_y_length=8.00,
    legend_display_type="CONTINUOUS",
    legend_border="OFF",
    legend_text_font_size=font_size,
    legend_text_colour="black",
#    legend_text_colour="white",
#    legend_title_text="River discharge (m3/s)",
    legend_title_text="River discharge (m3/s)   Inundated area (-)",
    legend_title_orientation="HORIZONTAL",
)

In [ ]:
pos_title = mv.mtext(text_line_1         = "IFS forecast of rivers and inundated area", text_mode           = "positional",
    text_box_x_position = 0, text_box_y_position = 17.4, text_box_x_length   = 8.2, text_box_y_length   = 1,
    text_font_size=font_size, text_colour="black")

pos_title2 = mv.mtext(text_line_1  = "Date "+datet, text_mode = "positional",
    text_box_x_position = 0, text_box_y_position = 16.8, text_box_x_length   = 7.5, text_box_y_length   = 1,
    text_font_size=font_size, text_colour="black")

notitle = mv.mtext(text_lines=[" "], text_font_size=font_size)

In [ ]:
y1=2020 ; m1=7 ; d1=22
y2=2020 ; m2=7 ; d2=22
delta=1

d0 = datetime.datetime(y1,m1,d1)
d1 = datetime.datetime(y2,m2,d2)
#d0 = datetime.datetime(2025,9,1)
#d1 = datetime.datetime(2025,9,5)
dt = datetime.timedelta(days = 1)
print (d0,d1,dt)
folder = "/perm/pad/flood_cases/"
outbasename = folder + "flood_" + flood_case
cnt = 0
ddt = 0
for d in np.arange(d0, d1, dt):
    for s in np.arange(delta, 27, delta):
        mars_request = {
        'class': 'rd',
        'expver': exp,
        'stream': stream,
        'type': 'fc',
        'levtype': 'sfc',
        'date': d,
        'time': time,
        'step': s,
        'param': parameter, 
        }
#         mars_request = {"class" : "rd", "date" : d, "expver" : exp, "stream" : "oper", "type" : "fc",
#         "levtype" : "sfc","step" : s,"param" : parameter,"time" : time,"area" : area}
        if os.path.exists(output):
            dd = mv.read(output)
            data = mv.read(data=dd,date=d,step=s)
        else:
            data = mv.retrieve(mars_request)
 #           t = mv.grib_get(data, ["dataDate"])[0][0]
 #           step = mv.grib_get(data, ["step"])[0][0]

        png = mv.png_output(output_name="{}_{:03d}".format(outbasename, cnt), 
        output_name_first_page_number="off",output_width=2000)
        mv.setoutput(png)
        dti=d0.strftime("%Y%m%d")
        st=int(s)/24
        tt=(st-int(st))*24.
        datet=str(int(dti)+ddt)
        datet=str(int(dti)+int(st))        
        timet=str(int(tt))
        datet=str(d)[:10]

        print (f"{datet} {timet} {outbasename} {cnt}")

        pos_title2 = mv.mtext(text_line_1  = "Date: "+datet+" "+timet+" UTC", text_mode = "positional",
        text_box_x_position = 0, text_box_y_position = 16.8, text_box_x_length   = 7.5, text_box_y_length   = 1,
        text_font_size=font_size, text_colour="black", )

        dw = mv.plot_superpage(pages=[mv.plot_page(view=view_area)])
 #       mv.plot(dw,data,conr,legend,notitle,pos_title,pos_title2)
        mv.plot(dw,data[0],conm,data[0],conr,data[1],Inundation, view_area, legend,notitle,pos_title,pos_title2)
        cnt +=  1
    ddt +=  1

In [ ]:
# ! convert -delay 20 -crop 848x844+120+230 +repage /perm/pad/flood_cases/river_*.png /perm/pad/flood_cases/river.gif

In [ ]:
! convert -delay 50 /perm/pad/flood_cases/flood_{flood_case}_0*.png flood_{flood_case}.gif

In [ ]:
rm -rf /perm/pad/flood_cases/flood_*.png

In [ ]:
output_max="/perm/pad/flood_cases/"+flood_case+"_flood_max.grb"
output_mean="/perm/pad/flood_cases/"+flood_case+"_flood_mean.grb"
print(output)

#d = mv.read(output)

max_field = data[1].max()
mean_field = data[1].mean()
max_field.write(output_max)
mean_field.write(output_mean)

In [ ]:
#End of Notebook. Note that the file _flood.grb will be used by the Benchmark Notebooks.
